### Initialize

In [78]:
import os
import json
import spotipy
from spotipy.cache_handler import CacheFileHandler
from spotipy.oauth2 import SpotifyOAuth

playset = dict()
known_tracks = dict()
known_artists = dict()
all_artists = set()

### (Re-)Authenticate user

In [83]:
cache_handler = CacheFileHandler(cache_path=".spotipy")
os.remove(cache_handler.cache_path)

auth_manager = SpotifyOAuth(
    scope="user-library-read playlist-read-private",
    client_id="",
    client_secret="",
    redirect_uri="http://localhost:8080",
    show_dialog=True,
    cache_handler=cache_handler,
)
sp = spotipy.Spotify(auth_manager=auth_manager)

current_user = sp.current_user()
print("Logged in as: " + current_user["display_name"])

Logged in as: Bobin Belloner


### Get user's tracks

In [82]:
known_tracks[current_user['id']] = set()
known_artists[current_user['id']] = set()

def add_track_to_playset(track):
    """Add a track to the playset and update the known tracks and artists"""
    playset[track['id']] = track
    known_tracks[current_user['id']].add(track['id'])
    for artist in track['artists']:
        known_artists[current_user['id']].add(artist['id'])
        all_artists.add(artist['id'])

# get liked tracks
results = sp.current_user_saved_tracks(limit=50)
while results:
    for item in results['items']:
        add_track_to_playset(item['track'])
    if results['next']:
        results = sp.next(results)
    else:
        results = None

# get tracks from users playlists
playlists = set()
results = sp.current_user_playlists(limit=50)
while results:
    for playlist in results['items']:
        playlists.add(playlist['id'])
    if results['next']:
        results = sp.next(results)
    else:
        results = None

for playlist in playlists:
    results = sp.playlist_tracks(playlist)
    while results:
        for item in results['items']:
            add_track_to_playset(item['track'])
        if results['next']:
            results = sp.next(results)
        else:
            results = None

### Filter and enrich playset

In [84]:
# Find the tracks all players know
k = 2 # how many players need to know a track
tracks_known_by_k = set()

for id, item in playset.items():
    known_by = 0
    for player in known_tracks.keys():
        if item['id'] in known_tracks[player]:
            known_by += 1
    if known_by >= k:
        tracks_known_by_k.add(item['id'])

playset = dict((k, v) for k, v in playset.items() if k in tracks_known_by_k)

In [85]:
# add tracks by artists that are known by k players
k = 2 # how many players need to know an artist
artists_known_by_k = set()

for artist in all_artists:
    known_by = 0
    for player in known_artists.keys():
        if artist in known_artists[player]:
            known_by += 1
    if known_by >= k:
        artists_known_by_k.add(artist)#

for artist in artists_known_by_k:
    results = sp.artist_top_tracks(artist, country='DE')
    tracks = results['tracks']
    for item in tracks:
        playset[item['id']] = item

### Output playset

In [100]:
with open('playset.json', 'w') as f:
    json.dump(list(playset.values()), f)

In [77]:
for item in playset.values():
    print(item['name'], ' - ', item['artists'][0]['name'])

atma  -  ok37
heat  -  ok37
Shaded  -  biskuwi
Dance With Me - Kölsch Remix  -  Kevin de Vries
Montezuma  -  RYSY
NRG  -  RYSY
Massive Bells  -  Cores
88  -  RYSY
Tokyo Nights  -  Kevin de Vries
Perception  -  Chris Avantgarde
Pegasus  -  Kevin de Vries
Motherboard  -  Ben Böhmer
Now Here No Where  -  Kölsch
Off Grid  -  Thomas Helmich
Reflector  -  Booka Shade
Healing  -  Massano
Green Karma - Original Mix  -  Cabaret Nocturne
HiTek  -  Insert Name Here
Blood Walk  -  Cabaret Nocturne
Slumber Party (feat. Princess Nokia)  -  Ashnikko
Worms  -  Ashnikko
STUPID (feat. Yung Baby Tate)  -  Ashnikko
Daisy  -  Ashnikko
Cheerleader  -  Ashnikko
Tantrum  -  Ashnikko
Invitation (feat. Kodie Shane)  -  Ashnikko
Deal With It (feat. Kelis)  -  Ashnikko
WEEDKILLER  -  Ashnikko
Daisy 2.0 (feat. Hatsune Miku)  -  Ashnikko
Toxic  -  Ashnikko
Moonlight Magic  -  Ashnikko
Hi, It's Me  -  Ashnikko
Maggots  -  Ashnikko
Cry (feat. Grimes)  -  Ashnikko
Miss Nectarine  -  Ashnikko
Super Soaker (feat. Daniel